In [0]:
# Read a sample of CSV files from the volume to understand the schema
df_sample = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/retail_q/volumes/blob_source/transactions_source/") \
    .limit(10)

display(df_sample)

## Auto Loader Bronze Ingestion

This notebook uses **Auto Loader** to incrementally load CSV files from the volume into a bronze table.

### How Auto Loader Works
* **Incremental Processing**: Only processes new or modified files
* **Checkpoint**: Tracks which files have been processed at `/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions`
* **Schema Evolution**: Automatically handles schema changes with `_rescued_data` column
* **Idempotent**: Running the same cell multiple times will NOT create duplicates

### Workflow
1. **First Time**: Run Cell 4 (Auto Loader) - loads all CSV files
2. **Subsequent Runs**: Run Cell 4 again - only processes NEW files added to the volume
3. **To Reprocess All Files**: Run Cell 3 to reset the checkpoint, then run Cell 4

In [0]:
%sql
-- Drop the table with typo (transactionss) if it exists
DROP TABLE IF EXISTS retail_q.blob_bronze.transactionss;

-- Also drop the old transactions table to start fresh with Auto Loader
DROP TABLE IF EXISTS retail_q.blob_bronze.transactions;

In [0]:
# Clean up checkpoint and schema directories for a fresh start
# ⚠️ ONLY run this when you want Auto Loader to reprocess ALL files
# Auto Loader uses checkpoints to track processed files - deleting them causes reprocessing

dbutils.fs.rm("/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions", True)
dbutils.fs.rm("/Volumes/retail_q/volumes/blob_source/_schemas/transactions", True)
print("✅ Checkpoint reset - Auto Loader will now reprocess all CSV files")

In [0]:
from pyspark.sql import functions as F

# Use Auto Loader to incrementally ingest CSV files
df_stream = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/retail_q/volumes/blob_source/_schemas/transactions") \
    .load("/Volumes/retail_q/volumes/blob_source/transactions_source/")

# Add metadata columns for data lineage
df_bronze = df_stream \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.col("_metadata.file_path"))

# Write to bronze table using Auto Loader (trigger once for batch-like behavior)
query = df_bronze.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("retail_q.blob_bronze.transactions")

# Wait for the stream to complete
query.awaitTermination()

print(f"Auto Loader completed successfully. Data loaded to retail_q.blob_bronze.transactions")

In [0]:
%sql
-- Verify the bronze table
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT transaction_id) as unique_transactions,
  MIN(ingestion_timestamp) as first_ingestion,
  MAX(ingestion_timestamp) as last_ingestion
FROM retail_q.blob_bronze.transactions